## 06. scratchpad

In [28]:
import pickle
import pandas as pd
import Data
import re
from collections import Counter
from tqdm.notebook import tqdm

In [3]:
# with open("../data/dataframe_eng.pkl", 'rb') as f:
#     data_eng = pickle.load(f)

# with open("../data/dataframe_pol.pkl", 'rb') as f:
#     data_pol = pickle.load(f)

In [2]:
# with open("../data/bpe_tokenizer_eng.pkl", 'rb') as f:
#     tokenizer_eng = pickle.load(f)
    
# with open("../data/bpe_tokenizer_pol.pkl", 'rb') as f:
#     tokenizer_pol = pickle.load(f)

### New data

In [4]:
with open('../data/europarl/europarl-v7.pl-en.en', 'r', encoding='utf-8') as f:
    eng_lines = f.readlines()

with open('../data/europarl/europarl-v7.pl-en.pl', 'r', encoding='utf-8') as f:
    pol_lines = f.readlines()

pairs = [(e.strip(), p.strip()) for e, p in zip(eng_lines, pol_lines)
         if e.strip() and p.strip() 
         and not e.strip().startswith('<') 
         and not p.strip().startswith('<')]

df_europarl = pd.DataFrame(pairs, columns=['eng_text', 'pol_text'])
print(len(df_europarl))
df_europarl.head()

629380


,eng_text,pol_text
0,Action taken on Parliament's resolutions: see ...,Działania podjęte w wyniku rezolucji Parlament...
1,Documents received: see Minutes,Składanie dokumentów: patrz protokół
2,Written statements (Rule 116): see Minutes,Oświadczenia pisemne (art. 116 Regulaminu): pa...
3,Texts of agreements forwarded by the Council: ...,Teksty porozumień przekazane przez Radę: patrz...
4,Membership of Parliament: see Minutes,Skład Parlamentu: patrz protokół


In [5]:
requir = lambda x: re.sub(r"^[(]..[)]", "", x)
df_europarl['eng_text'] = df_europarl['eng_text'].apply(requir)
df_europarl['pol_text'] = df_europarl['pol_text'].apply(requir)

In [6]:
mask_eng = df_europarl['eng_text'].apply(lambda x: bool(re.search(r".+[(].+[)].+", x)))
mask_pol = df_europarl['pol_text'].apply(lambda x: bool(re.search(r".+[(].+[)].+", x)))

In [7]:
mask_eng.sum(), mask_pol.sum(), (mask_eng | mask_pol).sum()

(22461, 21196, 25231)

In [8]:
df_europarl = df_europarl[~(mask_eng | mask_pol)].reset_index(drop=True)
df_europarl

,eng_text,pol_text
0,Action taken on Parliament's resolutions: see ...,Działania podjęte w wyniku rezolucji Parlament...
1,Documents received: see Minutes,Składanie dokumentów: patrz protokół
2,Texts of agreements forwarded by the Council: ...,Teksty porozumień przekazane przez Radę: patrz...
3,Membership of Parliament: see Minutes,Skład Parlamentu: patrz protokół
4,Membership of committees and delegations: see ...,Skład komisji i delegacji: patrz protokół
...,...,...
604144,Composition of committees and delegations : se...,Skład komisji i delegacji: Patrz protokól
604145,Agenda of the next sitting : see Minutes,Porządek obrad następnego posiedzenia: Patrz p...
604146,Closure of the sitting,Zamknięcie posiedzenia
604147,(The sitting closed at 22.25),(The sitting closed at 22.25)


In [9]:
uniq_eng = set("".join(df_europarl["eng_text"].astype(str)))
uniq_pol = set("".join(df_europarl["pol_text"].astype(str)))
print(f"Eng uniq: {len(uniq_eng)} | Pol uniq: {len(uniq_pol)}")

Eng uniq: 262 | Pol uniq: 267


In [10]:
tgt_eng = ['z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '"', '!', ' ', ';']
tgt_pol = ['ż', 'Ż', 'ź', 'Ź', 'ś', 'Ś', 'ń', 'Ń', 'ł', 'Ł', 'ę', 'Ę', 'ć', 'Ć', 'ą', 'Ą', 'ó', 'Ó', 'z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '"', '!', ' ', ';']

In [11]:
eng_delchars = [x for x in sorted(list(uniq_eng), reverse=True) if x not in tgt_eng]
pol_delchars = [x for x in sorted(list(uniq_pol), reverse=True) if x not in tgt_pol]
regex_eng = "[" + "".join(map(re.escape, eng_delchars)) + "]"
regex_pol = "[" + "".join(map(re.escape, pol_delchars)) + "]"

In [13]:
mask_eng = df_europarl["eng_text"].str.contains(regex_eng, na=False)
mask_pol = df_europarl["pol_text"].str.contains(regex_pol, na=False)
mask_eng.sum(), mask_pol.sum(), (mask_eng | mask_pol).sum()

(10115, 21667, 24257)

In [14]:
df_europarl = df_europarl[~(mask_eng | mask_pol)].reset_index(drop=True)
#df_europarl = df_europarl[~mask_eng].reset_index(drop=True)

In [15]:
uniq_eng = set("".join(df_europarl["eng_text"].astype(str)))
uniq_pol = set("".join(df_europarl["pol_text"].astype(str)))
print(f"Eng uniq: {len(uniq_eng)} | Pol uniq: {len(uniq_pol)}")

Eng uniq: 78 | Pol uniq: 94


In [17]:
mask_1 = (df_europarl["eng_text"].str[-1] == '.') & (df_europarl["pol_text"].str[-1] != '.')
mask_2 = (df_europarl["eng_text"].str[-1] != '.') & (df_europarl["pol_text"].str[-1] == '.')

In [18]:
mask_1.sum(), mask_2.sum(), (mask_1 | mask_2).sum()

(1960, 1108, 3068)

In [19]:
(~(mask_1 | mask_2)).sum()

576824

In [20]:
df_europarl = df_europarl[~(mask_1 | mask_2)].reset_index(drop=True)

In [21]:
def tokenize_eng(snt):
    snt = snt.lower()
    snt = re.sub("(?<! )'(?! )", r" '", snt)
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

def tokenize_pol(snt):
    snt = snt.lower()
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [22]:
df_europarl['eng_split'] = df_europarl["eng_text"].apply(tokenize_eng)
df_europarl['pol_split'] = df_europarl["pol_text"].apply(tokenize_pol)

In [23]:
df_europarl['eng_split']

0         [action, taken, on, parliament, 's, resolution...
1                    [documents, received, :, see, minutes]
2         [texts, of, agreements, forwarded, by, the, co...
3             [membership, of, parliament, :, see, minutes]
4         [membership, of, committees, and, delegations,...
                                ...                        
576819    [composition, of, committees, and, delegations...
576820    [agenda, of, the, next, sitting, :, see, minutes]
576821                          [closure, of, the, sitting]
576822          [(, the, sitting, closed, at, 22, ., 25, )]
576823                                               [., -]
Name: eng_split, Length: 576824, dtype: object

In [24]:
uniq_freq_eng = Counter(df_europarl['eng_split'].explode())
uniq_freq_pol = Counter(df_europarl['pol_split'].explode())

In [25]:
char_factor = lambda word: [x for x in word] + ['_']
vocab_chars_eng = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
vocab_chars_pol = [[char_factor(k), v] for k, v in uniq_freq_pol.most_common()]
print(len(vocab_chars_eng), vocab_chars_eng[:5], '\n')
print(len(vocab_chars_pol), vocab_chars_pol[:5]) 

50655 [[['t', 'h', 'e', '_'], 1019855], [[',', '_'], 712258], [['.', '_'], 562408], [['o', 'f', '_'], 488571], [['t', 'o', '_'], 458481]] 

158653 [[[',', '_'], 876176], [['.', '_'], 578767], [['w', '_'], 432544], [['i', '_'], 304762], [['n', 'a', '_'], 192303]]


In [26]:
pair_vocab_eng = [[list(zip(tab, tab[1:])), v] for tab, v in vocab_chars_eng]
pair_vocab_pol = [[list(zip(tab, tab[1:])), v] for tab, v in vocab_chars_pol]

In [29]:
from BPE_tokenizer import BPETokenizer

In [30]:
tokenizer_eng = BPETokenizer(vocab_chars_eng, pair_vocab_eng, 12000, 600, False)
tokenizer_eng.train_bpe()

  0%|          | 0/11945 [00:00<?, ?it/s]

In [32]:
tokenizer_pol = BPETokenizer(vocab_chars_pol, pair_vocab_pol, 24000, 600, True)
tokenizer_pol.train_bpe()

  0%|          | 0/23935 [00:00<?, ?it/s]

In [35]:
df_eng = df_europarl[['eng_text', 'eng_split']]
df_pol = df_europarl[['pol_text', 'pol_split']]

In [37]:
df_eng.head()

,eng_text,eng_split
0,Action taken on Parliament's resolutions: see ...,"[action, taken, on, parliament, 's, resolution..."
1,Documents received: see Minutes,"[documents, received, :, see, minutes]"
2,Texts of agreements forwarded by the Council: ...,"[texts, of, agreements, forwarded, by, the, co..."
3,Membership of Parliament: see Minutes,"[membership, of, parliament, :, see, minutes]"
4,Membership of committees and delegations: see ...,"[membership, of, committees, and, delegations,..."


In [41]:
encoder_eng1.vocab_encoder

{'<pad>': 0,
 '<unk>': 1,
 '<eos>': 2,
 '_': 3,
 'a': 4,
 'b': 5,
 '5': 6,
 'g': 7,
 '2': 8,
 'u': 9,
 's': 10,
 'x': 11,
 'n': 12,
 'y': 13,
 '3': 14,
 't': 15,
 ')': 16,
 "'": 17,
 ']': 18,
 '1': 19,
 '"': 20,
 'p': 21,
 'w': 22,
 'j': 23,
 '/': 24,
 'k': 25,
 '!': 26,
 'f': 27,
 '-': 28,
 'i': 29,
 '8': 30,
 'o': 31,
 'm': 32,
 '[': 33,
 '?': 34,
 'r': 35,
 'h': 36,
 'l': 37,
 '6': 38,
 '7': 39,
 '0': 40,
 '9': 41,
 '%': 42,
 ';': 43,
 '4': 44,
 'd': 45,
 'q': 46,
 'z': 47,
 '.': 48,
 'c': 49,
 ':': 50,
 'e': 51,
 '(': 52,
 'v': 53,
 ',': 54,
 'e_': 55,
 'th': 56,
 's_': 57,
 't_': 58,
 'n_': 59,
 'd_': 60,
 'the_': 61,
 'in': 62,
 'er': 63,
 'an': 64,
 'y_': 65,
 'ti': 66,
 're': 67,
 ',_': 68,
 'o_': 69,
 'or': 70,
 'en': 71,
 'al': 72,
 'on_': 73,
 '._': 74,
 'f_': 75,
 'on': 76,
 'ar': 77,
 'of_': 78,
 'ou': 79,
 'to_': 80,
 'and_': 81,
 'ro': 82,
 'is_': 83,
 'g_': 84,
 'in_': 85,
 'ing_': 86,
 'ed_': 87,
 'es_': 88,
 'is': 89,
 'at_': 90,
 'a_': 91,
 'er_': 92,
 'ic': 93,
 'at

In [48]:
class BPEEncoder_2():
    def __init__(self, bpe_vocab, thres_tup=55, is_tgt=False):
        self.vocab_encoder = {"".join(k): v for k, v in bpe_vocab.items()}
        self.vocab_tuple = list(bpe_vocab.keys())[thres_tup:]
        self.char_factor = lambda word: [x for x in word] + ['_']

    def encode_snt(self, tokenized_snt):
        return [y for x in tokenized_snt for y in self.encode_word(x)]

    def encode_word(self, word):
        word_factor = self.char_factor(word)
        word_id = self.vocab_encoder.get("".join(word_factor), None)
        if word_id is None:
            word_pairs = list(zip(word_factor, word_factor[1:]))
            for pair in self.vocab_tuple:
                if pair in word_pairs:
                    p_1, p_2 = map(re.escape, pair)
                    word_factor = re.sub(rf"(?<=\s){p_1}\s{p_2}(?=\s)", f"{pair[0]}{pair[1]}", f" {' '.join(word_factor)} ").split()
                    word_pairs = list(zip(word_factor, word_factor[1:]))
            yield from [self.vocab_encoder[x] for x in word_factor]
        else:
            yield from [word_id]

In [49]:
encoder_eng2 = BPEEncoder_2(tokenizer_eng.vocab)

In [51]:
encoder_eng2.encode_snt(['corrections', 'to', 'votes', 'and', 'voting', 'intentions', ':', 'see', 'minutes'])

[6556, 80, 3842, 81, 1976, 4494, 434, 790, 1791]

In [34]:
df_europarl.head()

,eng_text,pol_text,eng_split,pol_split
0,Action taken on Parliament's resolutions: see ...,Działania podjęte w wyniku rezolucji Parlament...,"[action, taken, on, parliament, 's, resolution...","[działania, podjęte, w, wyniku, rezolucji, par..."
1,Documents received: see Minutes,Składanie dokumentów: patrz protokół,"[documents, received, :, see, minutes]","[składanie, dokumentów, :, patrz, protokół]"
2,Texts of agreements forwarded by the Council: ...,Teksty porozumień przekazane przez Radę: patrz...,"[texts, of, agreements, forwarded, by, the, co...","[teksty, porozumień, przekazane, przez, radę, ..."
3,Membership of Parliament: see Minutes,Skład Parlamentu: patrz protokół,"[membership, of, parliament, :, see, minutes]","[skład, parlamentu, :, patrz, protokół]"
4,Membership of committees and delegations: see ...,Skład komisji i delegacji: patrz protokół,"[membership, of, committees, and, delegations,...","[skład, komisji, i, delegacji, :, patrz, proto..."


In [42]:
from tqdm.auto import tqdm
tqdm.pandas()

In [47]:
df_2 = df_europarl[['pol_text', 'pol_split']].copy()
df_2['pol_ids'] = df_2['pol_text'].progress_apply(encoder_pol.encode_snt)

  0%|          | 0/576824 [00:00<?, ?it/s]

In [48]:
with open("../data/dataframe_pol_europarl.pkl", 'wb') as f:
    pickle.dump(df_2, f)

In [3]:
with open("../data/tokenizer_eng_2.pkl", 'rb') as f:
    tokenizer_eng = pickle.load(f)

with open("../data/tokenizer_pol_2.pkl", 'rb') as f:
    tokenizer_pol = pickle.load(f)

with open("../data/dataframe_europarl.pkl", 'rb') as f:
    df_europarl = pickle.load(f)

In [49]:
with open("../data/tokenizer_eng_2.pkl", 'wb') as f:
    pickle.dump(tokenizer_eng, f)

with open("../data/tokenizer_pol_2.pkl", 'wb') as f:
    pickle.dump(tokenizer_pol, f)

In [50]:
with open("../data/dataframe_europarl.pkl", 'wb') as f:
    pickle.dump(df_europarl, f)